**The Relationship Between Film Runtime and IMDb Ratings**

In this project, I examine whether longer films tend to receive higher IMDb ratings and how this relationship differs between comedy and drama films. The analysis uses films released between 2000 and 2025 that received at least 1,000 IMDb votes. I visualize film runtimes and ratings and compare the relationship in the two genres using Spearman correlation.

In [ ]:
import matplotlib.pyplot as plt

runtimes = [85, 110, 160]
ratings = [7.4, 8.1, 6.9]

plt.scatter(runtimes, ratings)
plt.xlabel("Film runtime (minutes)")
plt.ylabel("Rating")
plt.show()

In [ ]:
import pandas as pd

url = "https://datasets.imdbws.com/title.ratings.tsv.gz"
imdb_ratings = pd.read_csv(url, sep="\t", compression="gzip")

imdb_ratings.head()

In [ ]:
import csv
import pandas as pd

url = "https://datasets.imdbws.com/title.basics.tsv.gz"
columns = [
    "tconst", "titleType", "primaryTitle", "startYear", "runtimeMinutes", "genres"
]

movie_chunks = []
for chunk in pd.read_csv(
    url,
    sep="\t",
    compression="gzip",
    usecols=columns,
    na_values=r"\N",
    dtype="string",
    quoting=csv.QUOTE_NONE,
    chunksize=200_000,
):
    chunk["startYear"] = pd.to_numeric(chunk["startYear"], errors="coerce")
    chunk["runtimeMinutes"] = pd.to_numeric(
        chunk["runtimeMinutes"], errors="coerce"
    )

    mask = (
        chunk["titleType"].eq("movie")
        & chunk["startYear"].between(2000, 2025)
        & chunk["runtimeMinutes"].gt(0)
    )
    movie_chunks.append(chunk.loc[mask])

movie_details = pd.concat(movie_chunks, ignore_index=True)
sufficient_votes = imdb_ratings[imdb_ratings["numVotes"] >= 1000]
movies = movie_details.merge(sufficient_votes, on="tconst")

print(f"Number of movies included in the analysis: {len(movies):,}")
movies[["primaryTitle", "runtimeMinutes", "averageRating", "genres"]].head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)

for ax, genre, title in zip(
    axes, ["Comedy", "Drama"], ["Comedy", "Drama"]
):
    mask = movies["genres"].str.contains(genre, na=False)
    group_data = movies[mask]

    ax.scatter(group_data["runtimeMinutes"], group_data["averageRating"], s=8, alpha=0.15)
    ax.set_title(f"{title} ({len(group_data):,} film)")
    ax.set_xlabel("Film runtime (minutes)")

axes[0].set_ylabel("IMDb rating")
plt.tight_layout()
plt.show()

In [ ]:
for genre, name in [("Comedy", "Comedy"), ("Drama", "Drama")]:
    group_data = movies[movies["genres"].str.contains(genre, na=False)]
    correlation = group_data["runtimeMinutes"].corr(group_data["averageRating"], method="spearman")
    print(f"{name}: {correlation:.2f}")

Among films released between 2000 and 2025 with at least 1,000 IMDb votes, I observed a positive relationship between runtime and rating for both comedy and drama. Spearman correlation was 0.28 for comedy and 0.30 for drama. These values describe the extent to which film runtime and rating rankings vary together. This analysis cannot determine how increasing a film's runtime would affect its rating.